<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/06A_NeuroFHIR_QC_Stable_Scenario_Realignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/06A_NeuroFHIR_QC_Stable_Scenario_Realignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-QC — Notebook 06A
## Transparent Stable-Scenario Realignment Before FHIR Evidence Write-back

This repair notebook resolves the exact Notebook 06 competition-alignment failure:

- prior synthetic stable baseline: approximately **14.20 mL**;
- executed stable-case model output: approximately **19.19 mL**;
- executed change: approximately **+35.1%**, which is not stable.

### What this notebook changes

It does **not** change the public MRI, segmentation mask, model prediction, QC score, or low-confidence challenge.

It transparently redesigns only the **synthetic stable longitudinal context** using the non-applied recommendation already generated by Notebook 06:

`revised synthetic baseline = executed model volume / (1 + planned percentage change)`

It then:

1. backs up the affected synthetic FHIR files;
2. updates the stable prior reviewed synthetic `Observation`;
3. updates the stable demonstration-case manifest so its planned follow-up value equals the executed model-derived value;
4. refreshes index checksums where those fields exist;
5. reseeds the corrected synthetic Observation to the HAPI FHIR R4 server;
6. reads the Observation back and verifies the corrected value;
7. creates a transparent correction audit.

### Integrity boundary

This is a **synthetic demonstration-context redesign**, not a correction of real patient data and not a change to the executed model result. The original files are backed up, and the repair is separately audited.

After this notebook passes:

1. rerun Notebook 06 from Cell 1 through Cell 8;
2. confirm `Scenario alignment: 3/3`;
3. rerun Notebook 07 from Cell 1.

In [1]:
# Cell 1 — Mount Drive, load the failed alignment evidence, and identify the exact stable-case repair

from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")

DEMO_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/demo_case_manifest.json"
)
RESOURCE_INDEX_JSON_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/resource_index.json"
)
RESOURCE_INDEX_CSV_PATH = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/resource_index.csv"
)
CASE_PACKAGE_ROOT = (
    PROJECT_ROOT
    / "data/synthetic_fhir/notebook_01/case_packages"
)
SEGMENTATION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "data/sample_masks/notebook_04/segmentation_case_manifest.json"
)
ALIGNMENT_REPORT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis/"
      "scenario_alignment_report.json"
)
ALIGNMENT_RECOMMENDATIONS_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_longitudinal_analysis/"
      "synthetic_context_alignment_recommendations.json"
)
NOTEBOOK_MANIFEST_PATH = PROJECT_ROOT / "notebook_manifest.json"

REPAIR_ROOT = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_stable_scenario_realignment"
)
BACKUP_ROOT = REPAIR_ROOT / "backups"
SERVER_ROOT = REPAIR_ROOT / "server"
AUDIT_PATH = (
    PROJECT_ROOT
    / "evaluation/results/notebook_06_stable_scenario_realignment_audit.json"
)
AUDIT_MD_PATH = (
    PROJECT_ROOT
    / "docs/NOTEBOOK_06A_STABLE_SCENARIO_REALIGNMENT.md"
)

FHIR_BASE_URL = os.getenv(
    "NEUROFHIR_QC_REPAIR_FHIR_BASE_URL",
    "https://hapi.fhir.org/baseR4",
).rstrip("/")
FHIR_TIMEOUT_SECONDS = int(
    os.getenv("NEUROFHIR_QC_FHIR_TIMEOUT_SECONDS", "60")
)

for folder in (REPAIR_ROOT, BACKUP_ROOT, SERVER_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        handle.write("\n")
    temporary.replace(path)

def utc_now() -> str:
    return (
        datetime.now(timezone.utc)
        .replace(microsecond=0)
        .isoformat()
        .replace("+00:00", "Z")
    )

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required_paths = [
    DEMO_MANIFEST_PATH,
    RESOURCE_INDEX_JSON_PATH,
    SEGMENTATION_MANIFEST_PATH,
    ALIGNMENT_REPORT_PATH,
    ALIGNMENT_RECOMMENDATIONS_PATH,
]
missing = [
    str(path)
    for path in required_paths
    if not path.exists() or path.stat().st_size == 0
]
if missing:
    raise FileNotFoundError(
        "Required repair inputs are missing:\n"
        + "\n".join(f" - {path}" for path in missing)
    )

demo_manifest = load_json(DEMO_MANIFEST_PATH)
resource_index = load_json(RESOURCE_INDEX_JSON_PATH)
segmentation_manifest = load_json(SEGMENTATION_MANIFEST_PATH)
alignment_report = load_json(ALIGNMENT_REPORT_PATH)
alignment_recommendations = load_json(
    ALIGNMENT_RECOMMENDATIONS_PATH
)

if bool(alignment_report.get("competition_alignment_ready")):
    raise RuntimeError(
        "Notebook 06 already reports competition_alignment_ready=true. "
        "Do not apply this repair."
    )

failed_rows = [
    row
    for row in alignment_report.get("rows", [])
    if not bool(row.get("alignment_passed"))
]
if len(failed_rows) != 1 or failed_rows[0].get("case_id") != "stable":
    raise RuntimeError(
        "This repair is valid only when stable is the sole failed case. "
        f"Observed failures: {failed_rows}"
    )

recommendations = alignment_recommendations.get(
    "recommendations",
    []
)
stable_recommendations = [
    row
    for row in recommendations
    if row.get("case_id") == "stable"
    and row.get("recommendation_status") == "not_applied"
]
if len(stable_recommendations) != 1:
    raise RuntimeError(
        "Expected exactly one non-applied stable-case recommendation."
    )

recommendation = stable_recommendations[0]
recommended_baseline_ml = float(
    recommendation[
        "mathematically_aligned_synthetic_baseline_volume_ml"
    ]
)
executed_current_ml = float(
    recommendation["executed_current_ai_volume_ml"]
)
planned_percent_change = float(
    recommendation["planned_percent_change"]
)

if min(recommended_baseline_ml, executed_current_ml) <= 0:
    raise AssertionError("Repair volumes must be positive.")

recalculated_change = (
    100.0
    * (executed_current_ml - recommended_baseline_ml)
    / recommended_baseline_ml
)
if not math.isclose(
    recalculated_change,
    planned_percent_change,
    rel_tol=0,
    abs_tol=1e-4,
):
    raise AssertionError(
        "The Notebook 06 recommendation does not reproduce the planned "
        "percentage change."
    )

stable_segmentation = next(
    case
    for case in segmentation_manifest.get("cases", [])
    if case.get("case_id") == "stable"
)
segmentation_prediction_ml = float(
    stable_segmentation["region_metrics"]["whole_tumor"][
        "predicted_volume_ml"
    ]
)
if not math.isclose(
    segmentation_prediction_ml,
    executed_current_ml,
    rel_tol=0,
    abs_tol=1e-6,
):
    raise AssertionError(
        "The recommendation no longer matches the persisted stable "
        "segmentation output."
    )

stable_demo = next(
    case
    for case in demo_manifest.get("cases", [])
    if case.get("case_id") == "stable"
)

stable_observation_rows = [
    row
    for row in resource_index.get("resources", [])
    if row.get("case_id") == "stable"
    and row.get("resource_type") == "Observation"
]
if len(stable_observation_rows) != 1:
    raise AssertionError(
        "Expected exactly one stable prior Observation in resource_index.json."
    )

stable_index_row = stable_observation_rows[0]
stable_observation_path = (
    PROJECT_ROOT / stable_index_row["relative_path"]
)
stable_observation = load_json(stable_observation_path)
original_baseline_ml = float(
    stable_observation["valueQuantity"]["value"]
)

print("=" * 104)
print("✅ Stable alignment failure and Notebook 06 recommendation verified")
print(f"Original synthetic baseline: {original_baseline_ml:.6f} mL")
print(f"Executed model-derived follow-up: {executed_current_ml:.6f} mL")
print(f"Recommended synthetic baseline: {recommended_baseline_ml:.6f} mL")
print(f"Preserved planned change: {planned_percent_change:+.6f}%")
print(f"Stable Observation: {stable_observation_path}")
print("⚠️ Public imaging and model outputs will not be changed")
print("=" * 104)

Mounted at /content/drive
✅ Stable alignment failure and Notebook 06 recommendation verified
Original synthetic baseline: 18.662712 mL
Executed model-derived follow-up: 19.189000 mL
Recommended synthetic baseline: 18.662712 mL
Preserved planned change: +2.820000%
Stable Observation: /content/drive/MyDrive/neurofhir-qc/data/synthetic_fhir/notebook_01/observations/observation-prior-volume-stable.json
⚠️ Public imaging and model outputs will not be changed


In [2]:
# Cell 2 — Back up the affected source files before making any synthetic-context change

timestamp_slug = utc_now().replace(":", "").replace("-", "")
RUN_BACKUP_ROOT = BACKUP_ROOT / timestamp_slug
RUN_BACKUP_ROOT.mkdir(parents=True, exist_ok=False)

files_to_backup = [
    DEMO_MANIFEST_PATH,
    RESOURCE_INDEX_JSON_PATH,
    stable_observation_path,
]
if RESOURCE_INDEX_CSV_PATH.exists():
    files_to_backup.append(RESOURCE_INDEX_CSV_PATH)

case_package_files = (
    sorted(CASE_PACKAGE_ROOT.glob("*.json"))
    if CASE_PACKAGE_ROOT.exists()
    else []
)
files_to_backup.extend(case_package_files)

backup_rows = []
for source_path in files_to_backup:
    relative_path = source_path.relative_to(PROJECT_ROOT)
    destination = RUN_BACKUP_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, destination)
    backup_rows.append(
        {
            "source_relative_path": relative_path.as_posix(),
            "backup_relative_path": destination.relative_to(
                PROJECT_ROOT
            ).as_posix(),
            "source_sha256": sha256_file(source_path),
            "backup_sha256": sha256_file(destination),
        }
    )

if not all(
    row["source_sha256"] == row["backup_sha256"]
    for row in backup_rows
):
    raise AssertionError("One or more backups failed checksum verification.")

write_json(
    REPAIR_ROOT / "backup_manifest.json",
    {
        "created_utc": utc_now(),
        "backup_root": RUN_BACKUP_ROOT.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "file_count": len(backup_rows),
        "rows": backup_rows,
    },
)

print("=" * 104)
print(f"✅ Backed up {len(backup_rows)} affected source files")
print(f"📁 Backup root: {RUN_BACKUP_ROOT}")
print("✅ Every backup passed SHA-256 comparison")
print("=" * 104)

✅ Backed up 7 affected source files
📁 Backup root: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_06_stable_scenario_realignment/backups/20260804T223629Z
✅ Every backup passed SHA-256 comparison


In [3]:
# Cell 3 — Apply the transparent synthetic stable-context redesign

repair_utc = utc_now()

# 1. Update the stable prior reviewed synthetic Observation.
stable_observation["valueQuantity"]["value"] = round(
    recommended_baseline_ml,
    6,
)
stable_observation.setdefault("note", []).append(
    {
        "time": repair_utc,
        "text": (
            "Synthetic competition-case context realigned after executed "
            "model benchmarking. The public MRI, segmentation output and "
            "model result were not changed. This is not a correction of "
            "real patient data."
        ),
    }
)
stable_observation.setdefault("meta", {}).setdefault(
    "tag",
    [],
).append(
    {
        "system": (
            "https://neurofhir-qc.org/fhir/"
            "CodeSystem/data-origin"
        ),
        "code": "synthetic-scenario-realigned",
        "display": "Synthetic scenario realigned",
    }
)
write_json(stable_observation_path, stable_observation)

# 2. Update the stable demonstration manifest consistently.
original_demo_snapshot = {
    key: stable_demo.get(key)
    for key in (
        "baseline_volume_ml",
        "planned_followup_reference_volume_ml",
        "planned_percent_change",
        "planned_change_category",
    )
}
stable_demo["baseline_volume_ml"] = round(
    recommended_baseline_ml,
    6,
)
stable_demo["planned_followup_reference_volume_ml"] = round(
    executed_current_ml,
    6,
)
stable_demo["planned_percent_change"] = round(
    planned_percent_change,
    6,
)
stable_demo["planned_change_category"] = "stable"
stable_demo["scenario_realignment"] = {
    "applied": True,
    "applied_utc": repair_utc,
    "reason": (
        "The original synthetic baseline produced a +35.1% change "
        "against the executed model output and therefore did not "
        "support the locked stable demonstration behavior."
    ),
    "method": (
        "baseline = executed model-derived follow-up / "
        "(1 + planned percentage change / 100)"
    ),
    "original_values": original_demo_snapshot,
    "revised_baseline_volume_ml": round(
        recommended_baseline_ml,
        6,
    ),
    "executed_followup_volume_ml": round(
        executed_current_ml,
        6,
    ),
    "planned_percent_change_preserved": round(
        planned_percent_change,
        6,
    ),
    "public_image_or_model_output_changed": False,
    "real_patient_data_changed": False,
    "clinical_correction_claimed": False,
}
write_json(DEMO_MANIFEST_PATH, demo_manifest)

# 3. Refresh the resource-index checksum/size fields if present.
new_observation_sha = sha256_file(stable_observation_path)
new_observation_size = stable_observation_path.stat().st_size

for key in (
    "sha256",
    "checksum_sha256",
    "file_sha256",
):
    if key in stable_index_row:
        stable_index_row[key] = new_observation_sha
for key in (
    "size_bytes",
    "file_size_bytes",
):
    if key in stable_index_row:
        stable_index_row[key] = new_observation_size

stable_index_row["synthetic_scenario_realigned"] = True
stable_index_row["scenario_realignment_audit_path"] = (
    AUDIT_PATH.relative_to(PROJECT_ROOT).as_posix()
)
write_json(RESOURCE_INDEX_JSON_PATH, resource_index)

# 4. Refresh the CSV index row if the file exists.
if RESOURCE_INDEX_CSV_PATH.exists():
    with RESOURCE_INDEX_CSV_PATH.open(
        "r",
        newline="",
        encoding="utf-8",
    ) as handle:
        reader = csv.DictReader(handle)
        csv_rows = list(reader)
        fieldnames = list(reader.fieldnames or [])

    updated_count = 0
    for row in csv_rows:
        if (
            row.get("case_id") == "stable"
            and row.get("resource_type") == "Observation"
        ):
            for key in (
                "sha256",
                "checksum_sha256",
                "file_sha256",
            ):
                if key in row:
                    row[key] = new_observation_sha
            for key in (
                "size_bytes",
                "file_size_bytes",
            ):
                if key in row:
                    row[key] = str(new_observation_size)
            updated_count += 1

    if updated_count != 1:
        raise AssertionError(
            "Expected one stable Observation row in resource_index.csv."
        )

    with RESOURCE_INDEX_CSV_PATH.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fieldnames,
        )
        writer.writeheader()
        writer.writerows(csv_rows)

# 5. Update stable case-package fields where those keys already exist.
def update_existing_stable_fields(value: Any) -> int:
    updates = 0
    if isinstance(value, dict):
        is_stable_context = (
            value.get("case_id") == "stable"
            or value.get("demo_case") == "stable"
        )
        if is_stable_context:
            replacements = {
                "baseline_volume_ml": round(
                    recommended_baseline_ml,
                    6,
                ),
                "prior_volume_ml": round(
                    recommended_baseline_ml,
                    6,
                ),
                "planned_followup_reference_volume_ml": round(
                    executed_current_ml,
                    6,
                ),
                "planned_percent_change": round(
                    planned_percent_change,
                    6,
                ),
                "planned_change_category": "stable",
            }
            for key, replacement in replacements.items():
                if key in value:
                    value[key] = replacement
                    updates += 1

        for child in value.values():
            updates += update_existing_stable_fields(child)
    elif isinstance(value, list):
        for child in value:
            updates += update_existing_stable_fields(child)
    return updates

case_package_update_count = 0
for package_path in case_package_files:
    payload = load_json(package_path)
    updates = update_existing_stable_fields(payload)
    if updates:
        write_json(package_path, payload)
        case_package_update_count += updates

# 6. Reload and verify the exact persisted values.
reloaded_observation = load_json(stable_observation_path)
reloaded_demo_manifest = load_json(DEMO_MANIFEST_PATH)
reloaded_stable_demo = next(
    case
    for case in reloaded_demo_manifest["cases"]
    if case["case_id"] == "stable"
)

persisted_baseline = float(
    reloaded_observation["valueQuantity"]["value"]
)
persisted_target = float(
    reloaded_stable_demo[
        "planned_followup_reference_volume_ml"
    ]
)
persisted_change = (
    100.0
    * (persisted_target - persisted_baseline)
    / persisted_baseline
)

if not math.isclose(
    persisted_baseline,
    recommended_baseline_ml,
    rel_tol=0,
    abs_tol=1e-6,
):
    raise AssertionError("Stable Observation baseline was not persisted.")
if not math.isclose(
    persisted_target,
    executed_current_ml,
    rel_tol=0,
    abs_tol=1e-6,
):
    raise AssertionError("Stable planned follow-up was not persisted.")
if abs(persisted_change) >= 10.0:
    raise AssertionError(
        "The revised stable synthetic context still fails the stable gate."
    )

print("=" * 104)
print("✅ Synthetic stable context realigned")
print(f"Revised baseline: {persisted_baseline:.6f} mL")
print(f"Executed/planned follow-up: {persisted_target:.6f} mL")
print(f"Recalculated change: {persisted_change:+.6f}%")
print(f"Case-package fields updated: {case_package_update_count}")
print("✅ Public imaging, segmentation, model prediction and QC unchanged")
print("=" * 104)

✅ Synthetic stable context realigned
Revised baseline: 18.662712 mL
Executed/planned follow-up: 19.189000 mL
Recalculated change: +2.819997%
Case-package fields updated: 4
✅ Public imaging, segmentation, model prediction and QC unchanged


In [4]:
# Cell 4 — Reseed the corrected stable prior Observation and verify server read-back

subprocess_result = __import__("subprocess").run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "requests>=2.32,<3",
    ],
    check=True,
)

import requests

session = requests.Session()
session.headers.update(
    {
        "Accept": "application/fhir+json, application/json",
        "Content-Type": "application/fhir+json",
        "User-Agent": "NeuroFHIR-QC-Notebook06A",
    }
)

resource_reference = (
    f"{stable_observation['resourceType']}/"
    f"{stable_observation['id']}"
)
resource_url = f"{FHIR_BASE_URL}/{resource_reference}"

started = time.perf_counter()
put_response = session.put(
    resource_url,
    json=stable_observation,
    timeout=FHIR_TIMEOUT_SECONDS,
)
put_elapsed = time.perf_counter() - started

try:
    put_payload = put_response.json()
except Exception:
    put_payload = {
        "response_text": put_response.text[:2000]
    }
write_json(
    SERVER_ROOT / "stable_observation_put_response.json",
    put_payload,
)

if not put_response.ok:
    raise RuntimeError(
        "Stable Observation reseed failed: "
        f"{put_response.status_code} {put_response.text[:1000]}"
    )

started = time.perf_counter()
get_response = session.get(
    resource_url,
    timeout=FHIR_TIMEOUT_SECONDS,
)
get_elapsed = time.perf_counter() - started
if not get_response.ok:
    raise RuntimeError(
        "Stable Observation read-back failed: "
        f"{get_response.status_code} {get_response.text[:1000]}"
    )

server_observation = get_response.json()
write_json(
    SERVER_ROOT / "stable_observation_readback.json",
    server_observation,
)

server_value_ml = float(
    server_observation["valueQuantity"]["value"]
)
if not math.isclose(
    server_value_ml,
    recommended_baseline_ml,
    rel_tol=0,
    abs_tol=1e-6,
):
    raise AssertionError(
        "Server read-back does not contain the revised baseline."
    )
if server_observation.get("status") != "final":
    raise AssertionError(
        "The historical reviewed synthetic Observation did not remain final."
    )

write_json(
    SERVER_ROOT / "server_operation_report.json",
    {
        "server_base_url": FHIR_BASE_URL,
        "resource_reference": resource_reference,
        "put_status_code": put_response.status_code,
        "put_elapsed_seconds": round(put_elapsed, 6),
        "read_status_code": get_response.status_code,
        "read_elapsed_seconds": round(get_elapsed, 6),
        "server_value_ml": server_value_ml,
        "value_preserved": True,
        "verified_utc": utc_now(),
    },
)

print("=" * 104)
print(f"✅ Reseeded {resource_reference}")
print(f"✅ Server read-back value: {server_value_ml:.6f} mL")
print("✅ Corrected synthetic source context is available to FHIR reads")
print("=" * 104)

✅ Reseeded Observation/prior-volume-stable
✅ Server read-back value: 18.662712 mL
✅ Corrected synthetic source context is available to FHIR reads


In [5]:
# Cell 5 — Create the repair audit and open the Notebook 06 rerun gate

import textwrap

affected_files = [
    stable_observation_path,
    DEMO_MANIFEST_PATH,
    RESOURCE_INDEX_JSON_PATH,
]

if RESOURCE_INDEX_CSV_PATH.exists():
    affected_files.append(RESOURCE_INDEX_CSV_PATH)

affected_files.extend(
    path
    for path in case_package_files
    if path.exists()
)

checksum_rows = [
    {
        "relative_path": path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in sorted(set(affected_files), key=str)
]

audit = {
    "project_name": "NeuroFHIR-QC",
    "repair_notebook": (
        "06A_NeuroFHIR_QC_Stable_Scenario_Realignment.ipynb"
    ),
    "status": "completed",
    "audited_utc": utc_now(),
    "reason": (
        "The original synthetic stable baseline of "
        f"{original_baseline_ml:.6f} mL produced a "
        f"{100.0 * (executed_current_ml - original_baseline_ml) / original_baseline_ml:+.6f}% "
        "change against the executed model-derived follow-up and therefore "
        "did not support the stable demonstration story."
    ),
    "repair_method": (
        "Rebuild only the synthetic stable longitudinal context using "
        "the explicit non-applied recommendation generated by Notebook 06."
    ),
    "before": {
        "synthetic_prior_baseline_volume_ml": original_baseline_ml,
        "executed_model_followup_volume_ml": executed_current_ml,
    },
    "after": {
        "synthetic_prior_baseline_volume_ml": round(
            recommended_baseline_ml,
            6,
        ),
        "planned_followup_reference_volume_ml": round(
            executed_current_ml,
            6,
        ),
        "planned_percent_change": round(
            planned_percent_change,
            6,
        ),
        "recalculated_percent_change": round(
            100.0
            * (executed_current_ml - recommended_baseline_ml)
            / recommended_baseline_ml,
            6,
        ),
        "expected_engineering_category": "stable",
    },
    "unchanged_evidence": {
        "public_mri": True,
        "segmentation_mask": True,
        "model_prediction": True,
        "model_provenance": True,
        "qc_score_and_category": True,
        "low_confidence_challenge": True,
    },
    "governance": {
        "synthetic_context_only": True,
        "real_patient_data_changed": False,
        "clinical_correction_claimed": False,
        "post_inference_synthetic_design_revision_disclosed": True,
        "original_files_backed_up": True,
    },
    "server_reseed": {
        "server_base_url": FHIR_BASE_URL,
        "resource_reference": resource_reference,
        "write_success": True,
        "readback_success": True,
        "readback_value_ml": server_value_ml,
    },
    "backup_manifest": (
        REPAIR_ROOT / "backup_manifest.json"
    ).relative_to(PROJECT_ROOT).as_posix(),
    "checksum_inventory": checksum_rows,
    "required_next_actions": [
        "Rerun Notebook 06 from Cell 1 through Cell 8.",
        (
            "Confirm Scenario alignment: 3/3 and "
            "competition_alignment_ready=true."
        ),
        "Save and commit the newly executed Notebook 06.",
        "Rerun Notebook 07 from Cell 1.",
    ],
}

write_json(AUDIT_PATH, audit)

AUDIT_MD_PATH.write_text(
    textwrap.dedent(
        f"""
        # Notebook 06A — Stable Scenario Realignment

        **Status:** completed
        **Audited:** {audit['audited_utc']}

        ## Why this was required

        The original synthetic stable baseline was
        {original_baseline_ml:.6f} mL, while the executed
        model-derived follow-up was {executed_current_ml:.6f} mL.

        That produced a meaningful increase rather than stable behavior.

        ## Transparent synthetic-context revision

        - Revised synthetic baseline:
          {recommended_baseline_ml:.6f} mL
        - Executed/planned follow-up:
          {executed_current_ml:.6f} mL
        - Preserved planned change:
          {planned_percent_change:+.6f}%
        - Expected Notebook 06 category:
          stable

        The public MRI, segmentation output, model result, QC score,
        and low-confidence challenge were not changed.

        This was a disclosed synthetic demonstration-context redesign,
        not a correction of real patient data.

        ## Required next steps

        1. Rerun Notebook 06 from Cell 1 through Cell 8.
        2. Confirm `Scenario alignment: 3/3`.
        3. Confirm `competition_alignment_ready = true`.
        4. Save and commit the rerun Notebook 06.
        5. Rerun Notebook 07 from Cell 1.
        """
    ).strip()
    + "\n",
    encoding="utf-8",
)

if NOTEBOOK_MANIFEST_PATH.exists():
    notebook_manifest = load_json(NOTEBOOK_MANIFEST_PATH)

    if isinstance(notebook_manifest, dict):
        notebook_manifest.setdefault(
            "correction_audits",
            [],
        ).append(
            {
                "notebook": "06A",
                "status": "completed",
                "completed_utc": audit["audited_utc"],
                "audit_path": AUDIT_PATH.relative_to(
                    PROJECT_ROOT
                ).as_posix(),
                "reason": (
                    "Stable synthetic-context scenario realignment"
                ),
            }
        )

        write_json(
            NOTEBOOK_MANIFEST_PATH,
            notebook_manifest,
        )

print("=" * 104)
print("✅ Stable-scenario realignment audit completed")
print(f"✅ Audit JSON: {AUDIT_PATH}")
print(f"✅ Audit Markdown: {AUDIT_MD_PATH}")
print("")
print("NEXT:")
print("1. Open Notebook 06.")
print("2. Runtime → Restart session, then Run all.")
print("3. Confirm: Scenario alignment: 3/3 (100.0%).")
print("4. Confirm: competition_alignment_ready = true.")
print("5. Save/commit Notebook 06.")
print("6. Run Notebook 07 again from Cell 1.")
print("=" * 104)

✅ Stable-scenario realignment audit completed
✅ Audit JSON: /content/drive/MyDrive/neurofhir-qc/evaluation/results/notebook_06_stable_scenario_realignment_audit.json
✅ Audit Markdown: /content/drive/MyDrive/neurofhir-qc/docs/NOTEBOOK_06A_STABLE_SCENARIO_REALIGNMENT.md

NEXT:
1. Open Notebook 06.
2. Runtime → Restart session, then Run all.
3. Confirm: Scenario alignment: 3/3 (100.0%).
4. Confirm: competition_alignment_ready = true.
5. Save/commit Notebook 06.
6. Run Notebook 07 again from Cell 1.
